# Module 2 · Demo — Your First Agent

**From 0 to Agentic AI — DataHack Summit 2026**

In the previous demo the model could only *request* a tool. Here we hand a prebuilt
agent a **real** tool — **web search** — and watch it answer a question end-to-end,
breaking the *can't act* wall from the last notebook.

> This is a **teaching demo**, not the project. We build the real Knowledge Assistant
> properly with LangGraph starting in **Module 4** — this is just to *see* an agent work.

### What it does
- Takes a question
- Decides whether it needs to **search the web**
- Calls the search tool, reads the results, and answers — **grounded**, with a trace we can inspect

---
## Setup

In [1]:
# Install the workshop stack (Colab). Locally, use `uv sync` instead.
# Version ranges match src/pyproject.toml (the single source of truth).
!pip install -q "langchain>=1.2,<2" "langchain-openai>=1.1,<2" \
               "langgraph>=1.0,<2" "langchain-tavily>=0.2"

In [2]:
import os
from getpass import getpass

# Local: load keys from src/.env. Colab: you'll be prompted for missing keys.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

# This demo needs an LLM key and a web-search key.
for key in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

---
## Step 1 · The web-search tool

`TavilySearch` is a ready-made LangChain tool built for LLMs — it returns clean search results the agent can read.

In [3]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(max_results=3)

# Quick sanity check — call the tool directly:
hits = web_search.invoke({"query": "What is LangGraph?"})
print(type(hits))
print(str(hits)[:400], "...")

<class 'dict'>
{'query': 'What is LangGraph?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.geeksforgeeks.org/machine-learning/what-is-langgraph', 'title': 'What is LangGraph - GeeksforGeeks', 'content': 'LangGraph is an open-source framework from LangChain designed to build and manage AI agent workflows using graph-based structures. It allows developers to define w ...


---
## Step 2 · Assemble the agent

A system prompt gives the agent its role; `create_agent` runs the loop.

In [4]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

llm = init_chat_model("gpt-4.1-mini", model_provider="openai", temperature=0)

SYSTEM_PROMPT = (
    "You are a helpful assistant for a software company. "
    "Answer questions about tools, services, and operations. "
    "Use web search when you need current or external information, "
    "and cite what you found. If unsure, say so."
)

agent = create_agent(
    llm,
    tools=[web_search],
    system_prompt=SYSTEM_PROMPT,
)

---
## Step 3 · Ask it something

A little helper to run a question and print the final answer.

In [5]:
def ask(question: str):
    result = agent.invoke({"messages": [("user", question)]})
    return result["messages"][-1].content, result

answer, result = ask("Our team is evaluating LangGraph. What is it, and what is its latest major version?")
print(answer)

LangGraph is an open-source framework created by LangChain for building, managing, and deploying complex generative AI agent workflows using graph-based structures. It allows developers to define workflows as nodes and edges, making complex agent interactions more structured, scalable, and easier to control. LangGraph provides a low-level orchestration framework for building stateful agents, enabling fine-grained control over agent workflows through a graph abstraction of nodes (functions), edges (control flow), and state (data passed around). It is designed to be production-ready and is used by companies like Uber, LinkedIn, Klarna, and Norwegian Cruise Line.

The latest major version of LangGraph is 1.0, which marks its first stable major release, focusing on stability and maturity rather than new features. This version was released around October 2025 and is considered production-ready with no breaking changes from previous iterations. 

If you need more detailed information or spec

### Inspect the trace
See the agent decide to search, read the results, then answer — the reason → act → observe loop.

In [6]:
for m in result['messages']:
    m.pretty_print()

================================ Human Message =================================

Our team is evaluating LangGraph. What is it, and what is its latest major version?
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_JZr3PAIgHNyVNBllrhdUavRV)
 Call ID: call_JZr3PAIgHNyVNBllrhdUavRV
  Args:
    query: What is LangGraph?
    search_depth: basic
================================= Tool Message =================================
Name: tavily_search

{"query": "What is LangGraph?", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.geeksforgeeks.org/machine-learning/what-is-langgraph", "title": "What is LangGraph - GeeksforGeeks", "content": "LangGraph is an open-source framework from LangChain designed to build and manage AI agent workflows using graph-based structures. It allows developers to define workflows as nodes and edges, making complex agent interactions more structured, scalable 

### Try your own
Swap in any question. Notice it only searches when it actually needs to.

In [7]:
answer, _ = ask("What does the term \"retrieval-augmented generation\" mean?")
print(answer)

Retrieval-augmented generation (RAG) is a technique in natural language processing and machine learning where a generative model, such as a language model, is enhanced by incorporating information retrieved from an external knowledge source or database. Instead of relying solely on the model's internal knowledge, the model retrieves relevant documents, passages, or data from a large corpus and uses this retrieved information to generate more accurate, informative, and contextually relevant responses or content.

In essence, RAG combines retrieval-based methods (which find and provide relevant information) with generative methods (which produce new text) to improve the quality and factual accuracy of generated outputs. This approach is particularly useful for tasks like question answering, summarization, and dialogue systems, where up-to-date or specialized knowledge is required beyond the training data of the generative model.


---
## What this agent can — and can't — do yet

✅ It **acts**: chooses and runs a tool, then answers from real results — the *can't act* wall, gone.

🚧 But it's still a black box, and two walls remain:
- ❌ We didn't **build the loop** — it's prebuilt. Module 3 opens it up by hand.
- ❌ No **internal knowledge** or **memory** yet — those come with the real build (Modules 4+).

Next we stop pressing the easy button and **understand** the loop it runs — the **ReAct pattern**.

---
## Key takeaways
- A prebuilt agent + one real tool (web search) already answers questions end-to-end.
- The message trace *is* the loop: request → run → observe → answer.
- This was a demo to *see* an agent work — the real Knowledge Assistant gets built on LangGraph from Module 4.